In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_curve, auc, f1_score, precision_score, recall_score
import os
from pathlib import Path

# Предпроцессинг

In [2]:
base_dir = Path.cwd().parent
path = os.path.join(base_dir, 'data', 'data.csv')

In [3]:
df = pd.read_csv(path)

In [4]:
df = df.sort_values(['serial_number', 'date']).reset_index(drop=True)

In [5]:
failed_serials = df[df['failure'] == 1]['serial_number'].unique()
healthy_serials = df[~df['serial_number'].isin(failed_serials)]['serial_number'].unique()

In [6]:
keep_fraction = 0.1  # Оставляем только 10% здоровых
selected_healthy = np.random.choice(healthy_serials,
                                    size=int(len(healthy_serials) * keep_fraction),
                                    replace=False)

In [7]:
df_balanced = df[df['serial_number'].isin(failed_serials) |
                 df['serial_number'].isin(selected_healthy)]

In [8]:
SMART_COLS = [
    'smart_1_raw', 'smart_2_raw', 'smart_3_raw', 'smart_5_raw',
    'smart_12_raw', 'smart_22_raw', 'smart_192_raw', 'smart_194_raw',
    'smart_197_raw', 'smart_198_raw', 'smart_199_raw'
]

In [9]:
df_balanced[SMART_COLS] = df_balanced.groupby('serial_number')[SMART_COLS].transform(lambda x: x.interpolate(limit_direction='both'))

/var/folders/f8/91jrsrf57yl88tk37jhrjch80000gn/T/ipykernel_20872/839805571.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_balanced[SMART_COLS] = df_balanced.groupby('serial_number')[SMART_COLS].transform(lambda x: x.interpolate(limit_direction='both'))


In [10]:
def create_deltas(df, sensor_columns):
    for col in sensor_columns:
        df[f'{col}_delta'] = df.groupby('serial_number')[col].diff().fillna(0)
    return df

In [11]:
def mark_risk_zone(df, days_before=3):
    df['target'] = df['failure']
    for sn, group in df.groupby('serial_number'):
        if group['failure'].any():
            last_idx = group.index[group['failure'] == 1][-1]
            start_idx = max(0, last_idx - days_before)
            df.loc[start_idx:last_idx, 'target'] = 1
    return df

In [12]:
def create_windows(df, feature_cols, target_col, window_size=7):
    X, y = [], []
    for sn, group in df.groupby('serial_number'):
        series = group[feature_cols].values
        labels = group[target_col].values
        if len(series) < window_size:
            continue
        for i in range(len(series) - window_size + 1):
            X.append(series[i : i + window_size])
            y.append(labels[i + window_size - 1])
    return np.array(X), np.array(y)

In [13]:
def normalize(df, mean, std, cols):
    df_norm = df.copy()
    df_norm[cols] = (df[cols] - mean) / std
    return df_norm

In [14]:
def split_data_by_serial(df, train_ratio=0.7, val_ratio=0.15):
    all_serials = df['serial_number'].unique()
    all_serials = np.random.permutation(all_serials)

    n_total = len(all_serials)
    train_end = int(n_total * train_ratio)
    val_end = train_end + int(n_total * val_ratio)

    train_serials = all_serials[:train_end]
    val_serials = all_serials[train_end:val_end]
    test_serials = all_serials[val_end:]

    train_df = df[df['serial_number'].isin(train_serials)].copy()
    val_df = df[df['serial_number'].isin(val_serials)].copy()
    test_df = df[df['serial_number'].isin(test_serials)].copy()
    return train_df, val_df, test_df

In [15]:
df_balanced = create_deltas(df_balanced, SMART_COLS)
df_balanced = mark_risk_zone(df_balanced, days_before=3)

/var/folders/f8/91jrsrf57yl88tk37jhrjch80000gn/T/ipykernel_20872/1188904499.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{col}_delta'] = df.groupby('serial_number')[col].diff().fillna(0)
/var/folders/f8/91jrsrf57yl88tk37jhrjch80000gn/T/ipykernel_20872/1188904499.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{col}_delta'] = df.groupby('serial_number')[col].diff().fillna(0)
/var/folders/f8/91jrsrf57yl88tk37jhrjch80000gn/T/ipykernel_20872/1188904499.py:3: SettingWithCopyWarning: 
A value

In [16]:
df_balanced.to_csv('data_prepared')

In [17]:
train_df, val_df, test_df = split_data_by_serial(df_balanced)

In [18]:
FEATURE_COLS = [f'{c}_delta' for c in SMART_COLS]  # SMART_COLS +

In [19]:
train_mean = train_df[FEATURE_COLS].mean()
train_std = train_df[FEATURE_COLS].std()

In [20]:
train_df = normalize(train_df, train_mean, train_std, FEATURE_COLS)
val_df = normalize(val_df, train_mean, train_std, FEATURE_COLS)
test_df = normalize(test_df, train_mean, train_std, FEATURE_COLS)

In [21]:
X_train, y_train = create_windows(train_df, FEATURE_COLS, 'target')
X_val, y_val = create_windows(val_df, FEATURE_COLS, 'target')
X_test, y_test = create_windows(test_df, FEATURE_COLS, 'target')

In [22]:
X_train.shape

(9940, 7, 11)

<!-- # LSTM -->

## Данные и модель

In [23]:
class HDDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [24]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout_rate, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim,
                            num_layers=num_layers,
                            dropout=dropout_rate,
                            batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        out = hn[-1]
        out = self.dropout(out)
        return self.fc(out)

In [25]:
BATCH_SIZE = 64

In [26]:
train_dataset = HDDataset(X_train, y_train)
val_dataset = HDDataset(X_val, y_val)
test_dataset = HDDataset(X_test, y_test)

In [27]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [28]:
criterion = nn.BCEWithLogitsLoss(weight=torch.tensor([5]))

In [29]:
model = LSTMModel(input_dim=11, hidden_dim=64, num_layers=2,dropout_rate=0.3, output_dim=1)

In [30]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

## Обучение и валидация

In [31]:
def evaluate(model, dataloader, threshold, criterion):
    model.eval()
    val_loss = 0
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            outputs = model(batch_x).squeeze()
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    preds = (all_probs > threshold).astype(int)
    f1 = f1_score(all_labels, preds)
    prec = precision_score(all_labels, preds, zero_division=0)
    rec = recall_score(all_labels, preds)
    precision_flow, recall_flow, _ = precision_recall_curve(all_labels, all_probs)
    pr_auc = auc(recall_flow, precision_flow)

    avg_loss = val_loss / len(dataloader)
    return avg_loss, f1, pr_auc, prec, rec

In [32]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=50):
    best_val_loss = float('inf')
    patience_counter = 0
    patience = 5

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        val_loss, f1, pr_auc, prec, rec = evaluate(model, val_loader, threshold=0.5, criterion=criterion)

        print(f"Epoch {epoch+1}/{epochs}\n",
              f"Train Loss: {train_loss/len(train_loader):.4f}\n",
              f"Val Loss: {val_loss:.4f}\n",
              f"PR-AUC: {pr_auc:.3f}\n",
              f"F1: {f1:.3f}\n"
              f"precision: {prec:.3f}\n",
              f"recall: {rec:.3f}\n")
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'LSTM_model.pth')
        else:
            patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping на эпохе {epoch+1}")
            break
    model.load_state_dict(torch.load('LSTM_model.pth'))
    return model


In [33]:
trained_model = train_model(model,train_loader, val_loader,criterion,optimizer)

Epoch 1/50
 Train Loss: 2.8861
 Val Loss: 1.1269
 PR-AUC: 0.080
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 2/50
 Train Loss: 0.6990
 Val Loss: 0.7306
 PR-AUC: 0.094
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 3/50
 Train Loss: 0.6485
 Val Loss: 0.7252
 PR-AUC: 0.095
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 4/50
 Train Loss: 0.6471
 Val Loss: 0.7206
 PR-AUC: 0.094
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 5/50
 Train Loss: 0.6407
 Val Loss: 0.7181
 PR-AUC: 0.093
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 6/50
 Train Loss: 0.6294
 Val Loss: 0.7133
 PR-AUC: 0.091
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 7/50
 Train Loss: 0.6192
 Val Loss: 0.7149
 PR-AUC: 0.091
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 8/50
 Train Loss: 0.6176
 Val Loss: 0.7148
 PR-AUC: 0.092
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 9/50
 Train Loss: 0.6215
 Val Loss: 0.7103
 PR-AUC: 0.092
 F1: 0.000
precision: 0.000
 recall: 0.000

Epoch 10/50
 Train Loss: 0.6

## Оценка модели

In [34]:
model.load_state_dict(torch.load('LSTM_model.pth'))

<All keys matched successfully>

In [35]:
model.eval()

LSTMModel(
  (lstm): LSTM(11, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

In [36]:
y_true = []
y_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)

        y_probs.extend(probs.cpu().numpy())
        y_true.extend(labels.numpy())

y_probs = np.array(y_probs).flatten()
y_true = np.array(y_true).flatten()

In [37]:
preds = (y_probs > 0.5).astype(int)
f1 = f1_score(y_true, preds)
prec = precision_score(y_true, preds, zero_division=0)
rec = recall_score(y_true, preds)
precision_flow, recall_flow, _ = precision_recall_curve(y_true, y_probs)
pr_auc = auc(recall_flow, precision_flow)

print(f"PR-AUC: {pr_auc:.3f}\n",
              f"F1: {f1:.3f}\n"
              f"precision: {prec:.3f}\n",
              f"recall: {rec:.3f}\n")

PR-AUC: 0.187
 F1: 0.000
precision: 0.000
 recall: 0.000



# Random forest

In [38]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [40]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

In [41]:
rf_model.fit(X_train_flat, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [42]:
y_probs = rf_model.predict_proba(X_test_flat)[:, 1]
threshold = 0.5
y_pred = (y_probs >= threshold).astype(int)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.98      0.94      0.96      1937
           1       0.19      0.42      0.26        72

    accuracy                           0.92      2009
   macro avg       0.59      0.68      0.61      2009
weighted avg       0.95      0.92      0.93      2009



In [43]:
f1 = f1_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred)
precision_flow, recall_flow, _ = precision_recall_curve(y_true, y_probs)
pr_auc = auc(recall_flow, precision_flow)

print(f"PR-AUC: {pr_auc:.3f}\n",
              f"F1: {f1:.3f}\n"
              f"precision: {prec:.3f}\n",
              f"recall: {rec:.3f}\n")

PR-AUC: 0.229
 F1: 0.264
precision: 0.194
 recall: 0.417



In [44]:
import pickle

In [45]:
pickle.dump(model, open("rf_model.pkl", "wb"))

# Catboost

In [46]:
from catboost import CatBoostClassifier

In [47]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
ratio = neg_count / pos_count

print(f"Вес положительного класса: {ratio:.2f}")

Вес положительного класса: 32.02


In [48]:
cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='AUC',
    scale_pos_weight=ratio,
    early_stopping_rounds=50,
    verbose=100,
    random_seed=42
)

In [49]:
cat_model.fit(
    X_train_flat, y_train,
    eval_set=(X_val_flat, y_val),
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.6294871	best: 0.6294871 (0)	total: 59ms	remaining: 59s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8478531403
bestIteration = 23

Shrink model to first 24 iterations.


In [50]:
y_probs = cat_model.predict_proba(X_test_flat)[:, 1]
threshold = 0.5
y_pred = (y_probs >= threshold).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.88      0.93      1937
           1       0.14      0.56      0.23        72

    accuracy                           0.87      2009
   macro avg       0.56      0.72      0.58      2009
weighted avg       0.95      0.87      0.90      2009



In [51]:
f1 = f1_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred)
precision_flow, recall_flow, _ = precision_recall_curve(y_true, y_probs)
pr_auc = auc(recall_flow, precision_flow)

print(f"PR-AUC: {pr_auc:.3f}\n",
              f"F1: {f1:.3f}\n"
              f"precision: {prec:.3f}\n",
              f"recall: {rec:.3f}\n")

PR-AUC: 0.249
 F1: 0.228
precision: 0.143
 recall: 0.556



# MLP

In [52]:
class HDDPerceptron(nn.Module):
    def __init__(self, input_dim):
        super(HDDPerceptron, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

In [53]:
model_mlp = HDDPerceptron(77)

In [54]:
optimizer_mlp = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

In [55]:
trained_model_mlp = train_model(model_mlp,train_loader, val_loader,criterion,optimizer_mlp)

Epoch 1/50
 Train Loss: 3.7595
 Val Loss: 3.7346
 PR-AUC: 0.036
 F1: 0.053
precision: 0.027
 recall: 0.824

Epoch 2/50
 Train Loss: 3.7609
 Val Loss: 3.7350
 PR-AUC: 0.039
 F1: 0.051
precision: 0.026
 recall: 0.794

Epoch 3/50
 Train Loss: 3.7612
 Val Loss: 3.7485
 PR-AUC: 0.039
 F1: 0.049
precision: 0.025
 recall: 0.765

Epoch 4/50
 Train Loss: 3.7679
 Val Loss: 3.7344
 PR-AUC: 0.038
 F1: 0.048
precision: 0.025
 recall: 0.735

Epoch 5/50
 Train Loss: 3.7607
 Val Loss: 3.7210
 PR-AUC: 0.038
 F1: 0.046
precision: 0.024
 recall: 0.706

Epoch 6/50
 Train Loss: 3.7632
 Val Loss: 3.7349
 PR-AUC: 0.037
 F1: 0.049
precision: 0.025
 recall: 0.750

Epoch 7/50
 Train Loss: 3.7618
 Val Loss: 3.7186
 PR-AUC: 0.036
 F1: 0.051
precision: 0.027
 recall: 0.794

Epoch 8/50
 Train Loss: 3.7636
 Val Loss: 3.7288
 PR-AUC: 0.034
 F1: 0.053
precision: 0.027
 recall: 0.824

Epoch 9/50
 Train Loss: 3.7662
 Val Loss: 3.7281
 PR-AUC: 0.038
 F1: 0.051
precision: 0.026
 recall: 0.779

Epoch 10/50
 Train Loss: 3.7

# Вывод

Нейронные сети показали хуже результаты, чем ансамблевые методы. Выбор остается в сторону Random Forest.